# FZP Cascade Comparison vs Element Number

This notebook plots focusing efficiency as a function of cascade depth \(N\) for three optics at the same aperture and aspect ratio (thickness / minimum feature size = 8):

- an **optimized diffractive cascade**
- a **cascade of Fresnel zone plates** whose focal lengths coincide at the target plane, with outer radii clipped to the cascade aperture
- a **single Fresnel zone plate** at the last-element focal length (independent of \(N\))

Load results produced by `paper/experiments/fzp_cascade_nelem_sweep.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

In [ ]:
matplotlib.rcParams['figure.dpi'] = 200
matplotlib.rcParams.update({'font.size': 24})

## Load sweep results

Prefer the stable untimestamped files written for this notebook. If those are missing, fall back to the latest timestamped run in `paper_data/`.

In [ ]:
path = repo_root / "paper_data"
prefix = "fzp_cascade_nelem_sweep"
params_path = path / f"{prefix}_params.npy"
results_path = path / f"{prefix}_results.npz"

if not params_path.exists() or not results_path.exists():
    param_candidates = sorted(path.glob(f"{prefix}_params_*.npy"))
    if not param_candidates:
        raise FileNotFoundError(
            f"No {prefix} results in {path}. Run paper/experiments/fzp_cascade_nelem_sweep.py first."
        )
    params_path = param_candidates[-1]
    results_path = path / params_path.name.replace("_params_", "_results_").replace(".npy", ".npz")

params = np.load(params_path, allow_pickle=True).item()
results = np.load(results_path, allow_pickle=True)
print(params_path.name)
print(results_path.name)

nelems = np.asarray(results["nelems"], dtype=int).reshape(-1)
opt_efficiencies = np.asarray(results["opt_efficiencies"], dtype=float).reshape(-1)
fzp_cascade_efficiencies = np.asarray(results["fzp_cascade_efficiencies"], dtype=float).reshape(-1)
fzp_efficiency = float(np.asarray(results["fzp_efficiency"]).reshape(-1)[0])
fzp_efficiencies = np.full(nelems.shape, fzp_efficiency, dtype=float)

colors = ["C0", "C1", (0.4, 0.651, 0.118)]

## Efficiency vs \(N\)

The single zone plate does not depend on cascade depth, so its efficiency is constant. The FZP cascade and the optimized cascade are re-designed at each \(N\).

In [ ]:
print(
    f"AR = {params['aspect_ratio']:.0f}, "
    f"E = {params['central_energy_ev']/1e3:.0f} keV, "
    f"single FZP efficiency = {fzp_efficiency:.4f}"
)
print(f"{'N':>4} {'opt. cascade':>14} {'FZP cascade':>14} {'single FZP':>14}")
for n, opt_e, casc_e, fzp_e in zip(
    nelems, opt_efficiencies, fzp_cascade_efficiencies, fzp_efficiencies
):
    print(f"{int(n):4d} {opt_e:14.4f} {casc_e:14.4f} {fzp_e:14.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    nelems, opt_efficiencies,
    "-o", linewidth=3, markersize=8, color=colors[0], label="optimized cascade",
)
ax.plot(
    nelems, fzp_cascade_efficiencies,
    "-o", linewidth=3, markersize=8, color=colors[1], label="FZP cascade",
)
ax.plot(
    nelems, fzp_efficiencies,
    "-o", linewidth=3, markersize=8, color=colors[2], label="single FZP",
)
ax.set_xlabel("N")
ax.set_ylabel("efficiency")
ax.set_xticks(nelems)
ax.legend(fontsize=16, frameon=False)
ax.grid(True, linewidth=1)
fig.tight_layout()